# Process Multiple Long Strips

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

import papermill

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [3]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,113.0,2025-06-05,GHL_pyapp_20250605T1621,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0403#4 0.392,0.392,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119,114.0,2025-06-05,GHL_pyapp_20250605T1625,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0403#1BluMark 0.382,0.382,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,115.0,2025-06-05,GHL_pyapp_20250605T1628,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0403#7 0.383,0.383,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
121,116.0,2025-06-05,GHL_pyapp_20250605T1633,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0402#14 0.384,0.384,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
#%% Filter The Tests To Process
# dfmasks = [
#     (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
#     dftests['Test name'] != 'GHL_pyapp_20250326T1258'
# ]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]
# dfmasks = [
#     dftests['Test date']=='2025-05-05',
# ]
dfmasks = [
    dftests['Test date']=='2025-06-05',
    #dftests['Test name']!='GHL_pyapp_20250512T',
    #dftests['ProcessingNotes'].str.startswith('loc runF'),
    #~dftests['Batch'].str.contains('Valve')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
74,69.0,2025-06-05,GHL_pyapp_20250605T1211,Jun5 0.325-0.349mg/mm bag ylw,BluePenES0323#16 0.335,0.335,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,70.0,2025-06-05,GHL_pyapp_20250605T1214,Jun5 0.325-0.349mg/mm bag ylw,ES0403#8 0.326,0.326,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,71.0,2025-06-05,GHL_pyapp_20250605T1218,Jun5 0.325-0.349mg/mm bag ylw,ES0402#11 0.342,0.342,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,72.0,2025-06-05,GHL_pyapp_20250605T1225,Jun5 0.325-0.349mg/mm bag ylw,ES0403#6 0.331,0.331,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,73.0,2025-06-05,GHL_pyapp_20250605T1228,Jun5 0.325-0.349mg/mm bag ylw,ES0403#19 0.329,0.329,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,74.0,2025-06-05,GHL_pyapp_20250605T1233,Jun5 0.325-0.349mg/mm bag ylw,ES0403#21 0.340,0.340,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,75.0,2025-06-05,GHL_pyapp_20250605T1301,Jun5 0.325-0.349mg/mm bag ylw,ES0403#30 0.345,0.345,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,76.0,2025-06-05,GHL_pyapp_20250605T1326,Jun5 0.325-0.349mg/mm bag ylw,ES0407#2 0.326,0.326,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,77.0,2025-06-05,GHL_pyapp_20250605T1330,Jun5 0.325-0.349mg/mm bag ylw,ES0331#31 0.334,0.334,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,78.0,2025-06-05,GHL_pyapp_20250605T1333,Jun5 0.325-0.349mg/mm bag ylw,ES0331#35 0.327,0.327,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);


STUDY: GHL_pyapp_20250605T1211
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1214
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1218
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1225
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1228
{   'study_has_an_ini_file': True,
    'study_has_json_info_f

In [6]:
# Master Parameters
oct_scalar_min = 30;
oct_scalar_max = 60;

In [7]:
#%% Load OCTSTUDY object, and start some processing on it
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print(octstudy)

    fname_merged_and_rescaled_volume = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    
    if(octstudy.study_info['study_num_oct_files']>0):
        # --Loading And Pre-Processing--
        if(fname_merged_and_rescaled_volume.exists()):
            pass;
            #print(f'Loading {fname_merged_and_rescaled_volume.name}')
            # Load the strip and merge into one volume
            #vdvol = vedo.Volume(pv.read(fname_merged_and_rescaled_volume));
            #octstudy.vdvol = vdvol;
        else:
            # Load OCT Data for this study
            octstudy.load_all_octs();


            # THESE WILL DO NOTHING IF ANTICIPATED OUTPUTS/ARTIFACTS ALREADY EXIST IN THE PROCESSED FOLDER

            # RGB Camera Images - Write them out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_individual_images(octstudy);

            # RGB Camera Images - Make a montage and write out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_montage_image(octstudy);
            #break;


            # IF MERGED STACKED AND RESCALED TO SCALAR RANGE FILE EXISTS, LOAD THAT; OTHERWISE PROCESS IT HERE
            octstudy.folder_study_processed.mkdir(exist_ok=True);
            fname = fname_merged_and_rescaled_volume;
            if(fname.exists()):
                print('Stacked volume byte-size .vtk file already exists, will not recreate.');
                print(fname);
            else:
                print(f'Generating merged and rescaled .vtk volume');
                # Generate merged and rescaled volume
                vdvol = octstudy.generate_merged_vdvol_and_rescaled(oct_scalar_min,oct_scalar_max);
                octstudy.vdvol = vdvol;
            
                # save this byte-adjusted volume
                vdvol.dataset.save(fname);
            
            # Unload OCT Data for this study (we will still keep the vdvol)
            octstudy.unload_all_octdata();

        # --Detailed Image Processing--
        if hasattr(octstudy,'vdvol'):
            del octstudy.vdvol;

<OCT_Study_Folder Object>
GHL_pyapp_20250605T1211 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
<OCT_Study_Folder Object>
GHL_pyapp_20250605T1214 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
<OCT_Study_Folder Object>
GHL_pyapp_20250605T1218 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
<OCT_St

# (util) delete some items from processed folder

In [ ]:
# Warning this can be destructive deleting processed data!!
if False:
    for idx,octstudy in enumerate(octstudies):
        print('~~~~~~~');
        print(octstudy.name);
        
        fname = (octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')
        if(fname.exists()):
            print('deleted',fname.name);
            fname.unlink();

        # flist = octstudy.folder_study_processed.glob('figout*');
        # for fname in flist:
        #     print('deleted',fname.name);
        #     fname.unlink();

        # flist = octstudy.folder_study_processed.glob('processing_step*');
        # for fname in flist:
        #     print('deleted',fname.name);
        #     fname.unlink();


~~~~~~~
GHL_pyapp_20250605T1211
deleted along_strip_data_extracted.hdf5
deleted figout_post_maze_test.mp4
deleted figout_post_maze_test_connection_at_1990px.jpg
deleted figout_post_maze_test_connection_at_2100px.jpg
deleted figout_post_maze_test_connection_at_2630px.jpg
deleted figout_post_maze_test_connection_at_3105px.jpg
deleted figout_post_maze_test_connection_at_315px.jpg
deleted figout_post_maze_test_connection_at_3415px.jpg
deleted figout_post_maze_test_connection_at_5610px.jpg
deleted figout_post_maze_test_connection_at_5615px.jpg
deleted figout_post_maze_test_connection_at_5620px.jpg
deleted figout_post_maze_test_connection_at_5655px.jpg
deleted figout_post_maze_test_connection_at_5660px.jpg
deleted figout_post_maze_test_connection_at_5680px.jpg
deleted figout_post_maze_test_connection_at_5685px.jpg
deleted figout_post_maze_test_connection_at_5720px.jpg
deleted figout_post_maze_test_connection_at_6865px.jpg
deleted figout_post_maze_test_connection_at_6935px.jpg
deleted figout_

# Call Notebooks - Processing A to D and E

In [9]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
norunlist = [
    #'GHL_pyapp_20250508T1112','GHL_pyapp_20250508T1128','GHL_pyapp_20250508T1134','GHL_pyapp_20250508T1143'
]
forcerunlist = [
    #'GHL_pyapp_20250508T1151',
    #'GHL_pyapp_20250508T1601',
    #'GHL_pyapp_20250508T1608',
]
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    rescheck = octstudy.resultsCheck();
    pp.pprint(rescheck)
    doWeRunTheNotebook = any([v is False for k,v in rescheck.items()])
    doWeRunTheNotebook = doWeRunTheNotebook or ('/dfstepE' not in rescheck['along_strip_data_extracted']);
    print('Run?',doWeRunTheNotebook)
    if((doWeRunTheNotebook and octstudy.name not in norunlist) or (octstudy.name in forcerunlist)):
        octstudies_to_run.append(octstudy);
    # if(octstudy.name in forcerunlist):
    #     octstudies_to_run.append(octstudy);

# if(len(octstudies_to_run)>4):
#     #octstudies_to_run=octstudies_to_run[0:4];
#     #octstudies_to_run=octstudies_to_run[-4:];
#     print('only a subset');
#     norunlist = [
#         'GHL_pyapp_20250508T1112','GHL_pyapp_20250508T1128','GHL_pyapp_20250508T1134','GHL_pyapp_20250508T1143'
#     ]
#     octstudies_to_run = [s for s in octstudies_to_run if s.name not in norunlist ]
#     pass;

#octstudies_to_run = octstudies_to_run[0:20];

print('~~~~~~~');
print('We will run the processing on {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run])

~~~~~~~
GHL_pyapp_20250605T1211
{   'along_strip_data_extracted': False,
    'exist_data_extracted': True,
    'figoutmp4_mazetest': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250605T1214
{   'along_strip_data_extracted': False,
    'exist_data_extracted': True,
    'figoutmp4_mazetest': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250605T1218
{   'along_strip_data_extracted': False,
    'exist_data_extracted': True,
    'figoutmp4_mazetest': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250605T1225
{   'along_strip_data_extracted': False,
    'exist_data_extracted': True,
    'figoutmp4_mazetest': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250605T1228
{   'along_strip_data_extracted': False,
    'exist_data_extracted': True,
    'figoutmp4_mazetest': False,
    'figoutmp4_stepA

# Prepare notebooks we will call

In [11]:
#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\nb_test_nbparams.ipynb";
#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250429_process_a_longstrip.ipynb";
nbpaths = [
    r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250606_process_a_longstrip.ipynb",
    r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250507_process_E_longstripmetric.ipynb"
];
for nbpath in nbpaths:
    print('Notebook:',Path(nbpath).name);
    parameters = papermill.inspect_notebook(nbpath)
    pp.pprint(parameters.keys());

Notebook: octproc_20250606_process_a_longstrip.ipynb
dict_keys(['oct_scalar_min', 'oct_scalar_max', 'codename', 'study_name', 'folder_octexport_root', 'folder_figure_temp'])
Notebook: octproc_20250507_process_E_longstripmetric.ipynb
dict_keys(['oct_scalar_min', 'oct_scalar_max', 'codename', 'folder_octexport_root', 'study_name', 'folder_figure_temp'])


# Call notebooks to process (Multiple-strips in parallel, concurrent futures)

In [12]:
import concurrent.futures

def run_notebook_on_an_octstudy(octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder):

    folder_temp_path = Path(r'D:\TEMP\OCTtmp');
    folder_temp_path.mkdir(parents=True,exist_ok=True);

    # run each notebook we have defined to be run
    nnotebooks = len(nbpaths);
    for count,nbpath in enumerate(nbpaths):
        nbpath = Path(nbpath)
        nbpath_out = folder_temp_path/(nbpath.stem+'_OUT_{:s}.ipynb').format(octstudy.name)

        # execute a notebook
        print(f'Launching for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} ...')
        try:
            papermill.execute_notebook(
                input_path=nbpath,
                output_path=nbpath_out,
                parameters=dict(
                    folder_octexport_root=folder_octexport_root.as_posix(),
                    codename = Path(nbpath).name,
                    study_name=octstudy.name,
                    folder_figure_temp=nbpath_out.parent.as_posix()
                )
            )
            print(f'Finished for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} !')
        except Exception as e:
            print(f'FAILED for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} !')
            break;
    print(f'alldone for {octstudy.name}')


# Using concurrent.futures to handle multiprocessing
def run_in_parallel(num_processes):
    # with concurrent.futures.ProcessPoolExecutor(max_workers=num_processes) as executor:
    #     futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
    #     for future in concurrent.futures.as_completed(futures):
    #         print(future.result())  # You can handle results or exceptions here
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
        for future in concurrent.futures.as_completed(futures):
            print(future.result());

# Example usage
num_processes = 8;  # Number of parallel processes
run_in_parallel(num_processes);

Launching for GHL_pyapp_20250605T1211 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...
Launching for GHL_pyapp_20250605T1214 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...
Launching for GHL_pyapp_20250605T1218 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...
Launching for GHL_pyapp_20250605T1225 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...
Launching for GHL_pyapp_20250605T1228 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...
Launching for GHL_pyapp_20250605T1233 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...
Launching for GHL_pyapp_20250605T1301 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...
Launching for GHL_pyapp_20250605T1326 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb ...


Executing:   0%|          | 0/53 [00:00<?, ?cell/s]

Executing:   0%|          | 0/53 [00:00<?, ?cell/s]

Executing:   0%|          | 0/53 [00:00<?, ?cell/s]

Executing:   0%|          | 0/53 [00:00<?, ?cell/s]

...

Finished for GHL_pyapp_20250605T1621 the 2/2 notebook octproc_20250507_process_E_longstripmetric.ipynb !
alldone for GHL_pyapp_20250605T1621
None
Finished for GHL_pyapp_20250605T1633 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb !
Launching for GHL_pyapp_20250605T1633 the 2/2 notebook octproc_20250507_process_E_longstripmetric.ipynb ...


Executing:   0%|          | 0/19 [00:00<?, ?cell/s]

Finished for GHL_pyapp_20250605T1636 the 1/2 notebook octproc_20250606_process_a_longstrip.ipynb !
Launching for GHL_pyapp_20250605T1636 the 2/2 notebook octproc_20250507_process_E_longstripmetric.ipynb ...


Executing:   0%|          | 0/19 [00:00<?, ?cell/s]

Finished for GHL_pyapp_20250605T1633 the 2/2 notebook octproc_20250507_process_E_longstripmetric.ipynb !
alldone for GHL_pyapp_20250605T1633
None
Finished for GHL_pyapp_20250605T1636 the 2/2 notebook octproc_20250507_process_E_longstripmetric.ipynb !
alldone for GHL_pyapp_20250605T1636
None


In [ ]:
import concurrent.futures
import time

def task(n):
    time.sleep(1)
    return n * n

# Using ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(task, i) for i in range(10)]
    for future in concurrent.futures.as_completed(futures):
        print(future.result())

# # Using ProcessPoolExecutor
# with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
#     results = executor.map(task, range(10))
#     for result in results:
#         print(result)

# Call Notebooks - Processing E

In [ ]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    rescheck = octstudy.resultsCheck();
    pp.pprint(rescheck);
    doWeRunTheNotebook = not any([v is False for k,v in rescheck.items()])
    doWeRunTheNotebook = doWeRunTheNotebook and ('/dfstepE' in rescheck['along_strip_data_extracted']);
    print('Run?',not doWeRunTheNotebook)
    if(not doWeRunTheNotebook):
        octstudies_to_run.append(octstudy);

print('~~~~~~~');
print('We will run the processing on {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run])

In [ ]:
# Prepare notebook to run
import papermill

#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\nb_test_nbparams.ipynb";
#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250429_process_a_longstrip.ipynb";
nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250507_process_E_longstripmetric.ipynb";

parameters = papermill.inspect_notebook(nbpath)
pp.pprint(parameters);

In [ ]:
# Call notebooks in parallel
import concurrent.futures

def run_notebook_on_an_octstudy(octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder):
    folder_temp_path = Path(r'D:\TEMP\OCTtmp');

    for nbpath in nbpaths:
        nbpath_out = folder_temp_path/(Path(nbpath).stem+'_OUT_{:s}.ipynb').format(octstudy.name)

    # set output path
    #nbpath_out = Path(nbpath).parent/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    nbpath_out = Path(r'C:\TEMP\OCTtmp')/(Path(nbpath).stem+'_OUT_{:s}.ipynb').format(octstudy.name)
    print(nbpath_out);

    # # execute a notebook
    papermill.execute_notebook(
        input_path=nbpath,
        output_path=nbpath_out,
        parameters=dict(codename = Path(nbpath).name,study_name=octstudy.name)
    )
    # print('start',octstudy.name);
    # time.sleep(1)
    # print('stop',octstudy.name)

# Using concurrent.futures to handle multiprocessing
def run_in_parallel(num_processes):
    # with concurrent.futures.ProcessPoolExecutor(max_workers=num_processes) as executor:
    #     futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
    #     for future in concurrent.futures.as_completed(futures):
    #         print(future.result())  # You can handle results or exceptions here
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
        for future in concurrent.futures.as_completed(futures):
            print(future.result())

# Example usage
num_processes = 4;  # Number of parallel processes
run_in_parallel(num_processes);

# Processing Step 0 - Replace VTK scalars with scaled bytes

In [ ]:
#del vdvol;
#del scalars_rescaled_as_int

# Develop VTK Range Rescaling and integer

In [ ]:
def doVtkImageMinMaxScaling(vtkimagedata):
    # restrict range and re-scale, cast to integer (will reduce memory by 4x)
    oct_scalar_min = 30;
    oct_scalar_max = 60;

    import vtk

    # Create VTK Pipeline Connections
    # 1. Thresholding Hi
    alg_thresholder = vtk.vtkImageThreshold();
    alg_thresholder.SetInputData(vtkimagedata)
    alg_thresholder.ThresholdByUpper(oct_scalar_min);
    alg_thresholder.SetOutValue(oct_scalar_min);
    alg_thresholder.ReplaceInOff();
    alg_thresholder.ReplaceOutOn();

    # 2. Thresholding Lo
    alg_thresholder2 = vtk.vtkImageThreshold();
    alg_thresholder2.SetInputConnection(alg_thresholder.GetOutputPort());
    alg_thresholder2.ThresholdByLower(oct_scalar_max);
    alg_thresholder2.SetOutValue(oct_scalar_max);
    alg_thresholder2.ReplaceInOff();
    alg_thresholder2.ReplaceOutOn();

    # 3. Subtract Minimum Value
    alg_math1 = vtk.vtkImageMathematics();
    alg_math1.SetInputConnection(alg_thresholder2.GetOutputPort());
    alg_math1.SetConstantC(-oct_scalar_min);
    alg_math1.SetOperationToAddConstant();

    # 4. Rescale to 0 to 255
    alg_math2 = vtk.vtkImageMathematics();
    alg_math2.SetInputConnection(alg_math1.GetOutputPort());
    alg_math2.SetConstantK(255.0/oct_scalar_min);
    alg_math2.SetOperationToMultiplyByK();
    #cast to unit8_t byte
    #alg_math2.SetOutputS();

    # 5. Cast to Uint8
    alg_caster = vtk.vtkImageCast();
    alg_caster.SetInputConnection(alg_math2.GetOutputPort());
    alg_caster.SetOutputScalarTypeToUnsignedChar();


    # execute the vtk pipline, wrap with pyvista, and return
    alg_final = alg_caster;
    alg_final.Update();

    final = pv.wrap(alg_final.GetOutput())
    return final;
mergedvol_rescaled = doVtkImageMinMaxScaling(mergedvol);

In [ ]:
np.max(mergedvol.active_scalars)

In [ ]:
print(np.min(mergedvol_rescaled.active_scalars),np.max(mergedvol_rescaled.active_scalars))

In [ ]:
mergedvol_rescaled['OCTintensity']

In [ ]:
del mergedvol,mergedvol_rescaled,alg_thresholder

In [ ]:
del alg_final